In [ ]:
# GRASP — Asignación de Boxes Ambulatorios · Hospital Grant Benavente

import re, sys, random, statistics, unicodedata
from math import gcd
from pathlib import Path
from functools import reduce
from collections import defaultdict
from datetime import time as dt_time, datetime as dt_datetime

import pandas as pd
import openpyxl
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter

# ── 1. Parámetros globales ─────────────────────────────────────────────────────

# Detecta la carpeta del script (modo .py) o del notebook (modo Jupyter).
# Los Excel deben estar en la misma carpeta que este archivo.
try:
    BASE_DIR = Path(__file__).parent.resolve()
except NameError:                          # Jupyter: __file__ no existe
    BASE_DIR = Path.cwd()

RUTA_EXCEL   = BASE_DIR / "Agendas_Hospital_Grant.xlsx"
RUTA_MAESTRO = BASE_DIR / "Excel_Maestro_Boxes.xlsx"

for _r in [RUTA_EXCEL, RUTA_MAESTRO]:
    if not _r.is_file():
        sys.exit(f"No se encontró '{_r.name}' en '{_r.parent}'. "
                 "Coloca los Excel en la misma carpeta que el notebook.")

W1, W2 = 100, 40
N      = 10
ALPHAS = [round(a * 0.1, 1) for a in range(11)]
DIA_INICIO_MIN, DIA_FIN_MIN = 8 * 60, 17 * 60
DIAS_VALIDOS = ["LUNES", "MARTES", "MIERCOLES", "JUEVES", "VIERNES"]
TIPO_MAP = {
    "CONSULTA EXAMEN FISICO":     "CEF",
    "CONSULTA SIN EXAMEN FISICO": "CSF",
    "PROCEDIMIENTO":              "PROC",
}
A_CODES = list(TIPO_MAP.values())
KEYWORD_CANON_MEDICO = {
    "CIRUJANO INFANTIL":        "CIRUJANO INFANTIL",
    "TRAUMATOLOGO INFANTIL":    "TRAUMATOLOGO INFANTIL",
    "TRAUMATOLOG":              "TRAUMATOLOGO INFANTIL",
    "ODONTOLOGO":               "ODONTOLOGO",
    "CIRUJANO MAXILO":          "CIRUJANO MAXILO-FACIAL",
    "MAXILO":                   "CIRUJANO MAXILO-FACIAL",
    "CIRUGIA INFANTIL Y ORTOP": "CIRUGIA INFANTIL Y ORTOPEDIA",
    "CIRUGIA INFANTIL":         "CIRUJANO INFANTIL",
}

# ── 2. Utilidades ──────────────────────────────────────────────────────────────

def normalize(text):
    return ''.join(c for c in unicodedata.normalize('NFD', str(text).upper().strip())
                   if unicodedata.category(c) != 'Mn')

def parse_tiempo(t):
    if t is None or pd.isna(t): return None
    if isinstance(t, (int, float)):
        f = float(t)
        if 0 <= f < 1: return int(round(f * 1440))
        if 0 <= f <= 24 and f == int(f): return int(f * 60)
    if isinstance(t, (dt_time, dt_datetime)): return t.hour * 60 + t.minute
    txt = str(t).strip()
    if not txt or txt.lower() == "nan" or ":" not in txt: return None
    try:
        p = txt.split(":"); return int(p[0]) * 60 + int(p[1])
    except Exception: return None

def min_to_block(m, delta): return (m - DIA_INICIO_MIN) // delta + 1
def _hhmm(m): return f"{m//60:02d}:{m%60:02d}"
def _safe_int(v, d=0):
    try:   return d if v is None or pd.isna(v) else int(v)
    except: return d

# ── 3. Carga del Excel Maestro ─────────────────────────────────────────────────

def load_maestro(filepath):
    # Hoja Boxes
    df_b = pd.read_excel(filepath, sheet_name="Boxes", header=2)
    df_b.columns = [str(c).strip() for c in df_b.columns]
    df_b = df_b[df_b["BOX_ID"].notna() & df_b["BOX_ID"].astype(str).str.startswith("BOX")]
    col_cef  = next((c for c in df_b.columns if c.startswith("CEF")),  "CEF")
    col_csf  = next((c for c in df_b.columns if c.startswith("CSF")),  "CSF")
    col_proc = next((c for c in df_b.columns if c.startswith("PROC")), "PROC")
    box_ids, cap_b, comba, labels = [], {}, {}, {}
    for _, row in df_b.iterrows():
        bid = str(row["BOX_ID"]).strip()
        if not bid: continue
        box_ids.append(bid)
        cap_b[bid]  = max(1, _safe_int(row.get("cap_b", 1), 1))
        labels[bid] = str(row.get("LABEL", bid)).strip() if not pd.isna(row.get("LABEL", bid)) else bid
        comba[(bid, "CEF")]  = _safe_int(row.get(col_cef,  0))
        comba[(bid, "CSF")]  = _safe_int(row.get(col_csf,  0))
        comba[(bid, "PROC")] = _safe_int(row.get(col_proc, 0))
    for b in box_ids:
        for a in A_CODES: comba.setdefault((b, a), 0)

    # Hoja Policlinicos
    df_p = pd.read_excel(filepath, sheet_name="Policlinicos", header=2)
    df_p.columns = [str(c).strip() for c in df_p.columns]
    df_p = df_p[df_p["POLICLINICO_ID"].notna() &
                ~df_p["POLICLINICO_ID"].astype(str).str.lower().str.startswith("nan")]
    nom_col = next((c for c in df_p.columns if "NOMBRE_ARCHIVO" in c), None)
    box_col = next((c for c in df_p.columns if "BOXES" in c), "BOXES (separados por coma)")
    poli_ids, b_poli, archivo_to_poli = [], {}, {}
    for _, row in df_p.iterrows():
        pid = str(row["POLICLINICO_ID"]).strip()
        if not pid: continue
        bxs = [b.strip() for b in str(row.get(box_col, "")).split(",") if b.strip()]
        nom = str(row.get(nom_col, "")).strip() if nom_col else ""
        poli_ids.append(pid); b_poli[pid] = bxs
        if nom and nom.lower() != "nan": archivo_to_poli[normalize(nom)] = pid
    box_poli_map = {b: pid for pid, bxs in b_poli.items() for b in bxs}

    # Hoja Habilitaciones_NM
    df_nm = pd.read_excel(filepath, sheet_name="Habilitaciones_NM", header=2)
    df_nm.columns = [str(c).strip() for c in df_nm.columns]
    df_nm = df_nm[df_nm["PROFESION_CANON"].notna() &
                  ~df_nm["PROFESION_CANON"].astype(str).str.lower().str.startswith("nan")]
    hab_box_nm, hab_poli_nm, kw_to_canon, prof_nm_set = {}, {}, {}, set()
    for _, row in df_nm.iterrows():
        canon = str(row["PROFESION_CANON"]).strip()
        if not canon: continue
        kws  = [k.strip() for k in str(row.get("KEYWORDS_DETECCION",      "")).split(",") if k.strip()]
        bxs  = [b.strip() for b in str(row.get("BOXES_HABILITADOS",       "")).split(",") if b.strip()]
        pols = [p.strip() for p in str(row.get("POLICLINICOS_HABILITADOS", "")).split(",") if p.strip()]
        hab_box_nm[canon] = bxs; hab_poli_nm[canon] = pols; prof_nm_set.add(canon)
        for kw in kws: kw_to_canon[normalize(kw)] = canon

    # Hoja Especialidades_Poli
    df_e = pd.read_excel(filepath, sheet_name="Especialidades_Poli", header=2)
    df_e.columns = [str(c).strip() for c in df_e.columns]
    df_e = df_e[df_e["ESPECIALIDAD_CANON"].notna() &
                ~df_e["ESPECIALIDAD_CANON"].astype(str).str.lower().str.startswith("nan")]
    esp_poli, especialidades_canon = {}, set()
    for _, row in df_e.iterrows():
        esp = normalize(str(row["ESPECIALIDAD_CANON"]).strip())
        if not esp: continue
        especialidades_canon.add(esp)
        for p in [x.strip() for x in str(row.get("POLICLINICOS_HABILITADOS","")).split(",") if x.strip()]:
            esp_poli[(esp, p)] = 1

    # Hoja Reglas_Especiales
    df_r = pd.read_excel(filepath, sheet_name="Reglas_Especiales", header=2)
    df_r.columns = [str(c).strip() for c in df_r.columns]
    df_r = df_r[df_r["TIPO"].isin({"BOX_EXCLUSIVO", "BOX_FORZADO"})]
    reglas = []
    for _, row in df_r.iterrows():
        rid = str(row.get("ID",   "")).strip()
        tp  = str(row.get("TIPO", "")).strip()
        pn  = normalize(str(row.get("PROFESIONAL",    "")).strip())
        an  = normalize(str(row.get("ACTIVIDAD_NORM", "*")).strip())
        if an in {"", "NAN"}: an = "*"
        box = str(row.get("BOX_FORZADO", "")).strip()
        if rid and tp and pn and box:
            reglas.append({
                "id": rid, "tipo": tp, "prof_norm_regla": pn,
                "dia": normalize(str(row.get("DIA", "")).strip()),
                "hd": row.get("HORA_DESDE"), "hh": row.get("HORA_HASTA"),
                "actividad_norm": an, "box": box,
                "descripcion": str(row.get("DESCRIPCION","")).strip() if "DESCRIPCION" in df_r.columns else "",
            })

    boxes_excl_nm    = set(b for bxs in hab_box_nm.values() for b in bxs)
    boxes_excl_regla = {r["box"] for r in reglas if r["tipo"] == "BOX_EXCLUSIVO"}
    b_medicos = [b for b in box_ids if b not in boxes_excl_nm and b not in boxes_excl_regla]

    print(f"\n  Maestro | {len(box_ids)} boxes ({len(b_medicos)} médicos) | polis: {poli_ids}"
          f" | NM: {sorted(prof_nm_set)} | reglas: {[r['id'] for r in reglas]}")
    return {
        "box_ids": box_ids, "cap_b": cap_b, "labels": labels, "comba": comba,
        "b_medicos": b_medicos, "boxes_excl_nm": boxes_excl_nm, "boxes_excl_regla": boxes_excl_regla,
        "poli_ids": poli_ids, "b_poli": b_poli, "box_poli_map": box_poli_map,
        "archivo_to_poli": archivo_to_poli, "hab_box_nm": hab_box_nm, "hab_poli_nm": hab_poli_nm,
        "kw_to_canon": kw_to_canon, "prof_nm_set": prof_nm_set,
        "esp_poli": esp_poli, "especialidades_canon": especialidades_canon, "reglas": reglas,
    }

def detect_nm_canon(profesion_norm, kw_to_canon):
    for kw, canon in kw_to_canon.items():
        if kw in profesion_norm: return canon
    return None

# ── 4. Reglas especiales ───────────────────────────────────────────────────────

def build_regla_map(reglas, name_norm_to_prof, delta):
    regla_map = []
    for r in reglas:
        pn = r["prof_norm_regla"]
        prof_real = name_norm_to_prof.get(pn)
        if prof_real is None:
            print(f"  [AVISO] Regla {r['id']}: '{pn}' no en agenda."); continue
        hd, hh = parse_tiempo(r["hd"]), parse_tiempo(r["hh"])
        if hd is None or hh is None or hh <= hd:
            print(f"  [AVISO] Regla {r['id']}: horario inválido."); continue
        regla_map.append({**r, "profesional": prof_real, "prof_norm": pn,
                           "blocks": set(range(min_to_block(hd, delta), min_to_block(hh, delta)))})
    return regla_map

def get_box_regla(prof_norm, a_code, actividad, day, bloques, regla_map, comba):
    act_norm, day_norm = normalize(actividad), normalize(day)
    for r in regla_map:
        if r["prof_norm"] != prof_norm: continue
        if r["dia"] not in {"*", day_norm}: continue
        if not (set(bloques) & r["blocks"]): continue
        ra = r.get("actividad_norm", "*")
        if ra != "*" and ra not in act_norm: continue
        if comba.get((r["box"], a_code), 0) == 1:
            return r["box"], r["tipo"]
    return None, None

# ── 5. Lectura de agenda ───────────────────────────────────────────────────────

def leer_agenda(ruta, maestro):
    kw_to_canon     = maestro["kw_to_canon"]
    poli_ids        = maestro["poli_ids"]
    archivo_to_poli = maestro["archivo_to_poli"]

    poli_default = archivo_to_poli.get(normalize(Path(ruta).stem))
    if poli_default:
        print(f"  Policlínico desde archivo: '{poli_default}'")
    else:
        poli_default = poli_ids[0] if poli_ids else None
        print(f"  [AVISO] Policlínico no detectado, usando: '{poli_default}'")

    xl = pd.ExcelFile(ruta)
    raws, durs = [], []

    for sheet in xl.sheet_names:
        df = pd.read_excel(ruta, sheet_name=sheet, header=None)
        nombre = profesion = subesp = amb_hours = None
        policlinico = poli_default

        for i in range(min(25, len(df))):
            for j in range(df.shape[1]):
                vj   = str(df.iloc[i, j]) if not pd.isna(df.iloc[i, j]) else ""
                vj_n = normalize(vj)
                if "POLICLINICO" in vj_n and ":" in vj_n:
                    rp = vj_n.split("POLICLINICO:")[-1].strip()
                    m  = next((p for p in poli_ids if normalize(p) == rp or rp in normalize(p)), None)
                    if m: policlinico = m
                if "HORAS DE ATENCION AMBULATORIA" in vj_n:
                    nums = re.findall(r"[\d,\.]+", vj)
                    if nums:
                        try: amb_hours = float(nums[-1].replace(",", "."))
                        except ValueError: pass
            v0   = str(df.iloc[i, 0]) if not pd.isna(df.iloc[i, 0]) else ""
            v0_n = normalize(v0)
            if "NOMBRE PROFESIONAL:" in v0_n and nombre    is None: nombre    = v0.split(":", 1)[-1].strip()
            if "PROFESION:"          in v0_n and profesion is None: profesion = normalize(v0.split(":", 1)[-1].strip())
            if "SUBESPECIALIDAD:"    in v0_n and subesp    is None: subesp    = normalize(v0.split(":", 1)[-1].strip())

        if nombre is None: continue

        nm_canon = detect_nm_canon(profesion or "", kw_to_canon)
        is_nm    = nm_canon is not None
        if is_nm:
            specialty = nm_canon
            hab_boxes = maestro["hab_box_nm"].get(nm_canon, [])
        else:
            specialty = next((c for kw, c in KEYWORD_CANON_MEDICO.items() if subesp and kw in subesp),
                             "CIRUJANO INFANTIL")
            hab_boxes = []
        palt = 1 if (amb_hours is not None and amb_hours > 22) else 0

        hdr = None
        for i, row in df.iterrows():
            c0 = normalize(str(row.iloc[0])) if not pd.isna(row.iloc[0]) else ""
            c1 = normalize(str(row.iloc[1])) if df.shape[1] > 1 and not pd.isna(row.iloc[1]) else ""
            if "DIA" in c0 and "ACTIVIDAD" in c1: hdr = i; break
        if hdr is None: continue

        dia_actual = None
        for idx in range(hdr + 2, len(df)):
            row  = df.iloc[idx]
            dv   = normalize(str(row.iloc[0]).strip()) if not pd.isna(row.iloc[0]) else ""
            if dv in DIAS_VALIDOS: dia_actual = dv
            if not dia_actual: continue
            act      = str(row.iloc[1]).strip() if not pd.isna(row.iloc[1]) else ""
            tipo_raw = str(row.iloc[2]).strip() if not pd.isna(row.iloc[2]) else ""
            tipo     = normalize(tipo_raw)
            if "NOMBRE" in normalize(act) or "FIRMA" in normalize(act): break
            if tipo not in TIPO_MAP or not act or act.lower() == "nan": continue
            d_min = parse_tiempo(row.iloc[3])
            h_min = parse_tiempo(row.iloc[4])
            if d_min is None or h_min is None or h_min <= d_min: continue
            if d_min < DIA_INICIO_MIN or h_min > DIA_FIN_MIN:
                print(f"  [AVISO] Horario fuera 08-17: {nombre}|{dia_actual}|{_hhmm(d_min)}-{_hhmm(h_min)}")
            durs.append(h_min - d_min)
            raws.append({
                "profesional": nombre, "profesional_norm": normalize(nombre),
                "profesion": profesion or "", "specialty": specialty,
                "es_no_medico": is_nm, "nm_canon": nm_canon, "hab_boxes": hab_boxes,
                "policlinico": policlinico, "palt": palt,
                "tipo": TIPO_MAP[tipo], "actividad": act, "actividad_norm": normalize(act),
                "dia": dia_actual, "desde_min": d_min, "hasta_min": h_min,
            })

    if not durs: sys.exit("No se encontraron actividades válidas.")
    delta = reduce(gcd, durs)
    actividades, Palt = [], {}
    CAMPOS = ["profesional","profesional_norm","profesion","specialty","es_no_medico",
              "nm_canon","hab_boxes","policlinico","palt","tipo","actividad",
              "actividad_norm","dia","desde_min","hasta_min"]
    for i, r in enumerate(raws):
        t_ini = min_to_block(r["desde_min"], delta)
        t_fin = min_to_block(r["hasta_min"], delta)
        actividades.append({"id": i, **{k: r[k] for k in CAMPOS},
                             "bloques": list(range(t_ini, t_fin)), "n_bloques": t_fin - t_ini})
        Palt[r["profesional"]] = r["palt"]

    P_M  = sorted({a["profesional"] for a in actividades if not a["es_no_medico"]})
    P_NM = sorted({a["profesional"] for a in actividades if a["es_no_medico"]})
    name_norm_to_prof = {a["profesional_norm"]: a["profesional"] for a in actividades}
    return actividades, Palt, delta, P_M, P_NM, name_norm_to_prof

# ── 6. Ocupación y factibilidad ────────────────────────────────────────────────

def disponible(box, a, ocupacion, cap_b):
    prof, dia = a["profesional"], a["dia"]
    for t in a["bloques"]:
        if ocupacion.get((dia, t, "box",  box),  0)      >= cap_b.get(box, 1): return False
        if ocupacion.get((dia, t, "prof", prof), False): return False
    return True

def ocupar(box, a, ocupacion):
    prof, dia = a["profesional"], a["dia"]
    for t in a["bloques"]:
        ocupacion[(dia, t, "box",  box)]  = ocupacion.get((dia, t, "box", box), 0) + 1
        ocupacion[(dia, t, "prof", prof)] = True

def liberar(box, a, ocupacion):
    prof, dia = a["profesional"], a["dia"]
    for t in a["bloques"]:
        cnt = ocupacion.get((dia, t, "box", box), 0)
        if cnt > 1: ocupacion[(dia, t, "box", box)] = cnt - 1
        else:       ocupacion.pop((dia, t, "box", box), None)
        ocupacion.pop((dia, t, "prof", prof), None)

def reconstruir_ocupacion(asign, actividades):
    ocu = {}
    for a in actividades:
        box = asign.get(a["id"])
        if box is not None: ocupar(box, a, ocu)
    return ocu

# ── 7. Función objetivo Z ──────────────────────────────────────────────────────

def _box_por_bloque(asign, acts_dia):
    mapa = {}
    for a in acts_dia:
        box = asign.get(a["id"])
        if box is not None:
            for t in a["bloques"]: mapa[(a["profesional"], t)] = box
    return mapa

def costo_estabilidad(asign, acts_dia, Palt):
    mapa = _box_por_bloque(asign, acts_dia)
    if not mapa: return 0.0
    bloques = sorted({t for (_, t) in mapa.keys()})
    costo = 0.0
    for prof in {a["profesional"] for a in acts_dia}:
        w = W2 * (1 + Palt.get(prof, 0))
        for t in range(bloques[0], bloques[-1]):
            b1, b2 = mapa.get((prof, t)), mapa.get((prof, t + 1))
            if b1 is not None and b2 is not None and b1 != b2: costo += w
    return costo

def calcular_Z(asign, actividades, Palt):
    pen_u = sum(a["n_bloques"] for a in actividades if asign.get(a["id"]) is None)
    acts_dia = defaultdict(list)
    for a in actividades: acts_dia[a["dia"]].append(a)
    return -W1 * pen_u - sum(costo_estabilidad(asign, v, Palt) for v in acts_dia.values())

def desglose_Z(asign, actividades, Palt):
    b_sin = sum(a["n_bloques"] for a in actividades if asign.get(a["id"]) is None)
    pen_u = W1 * b_sin
    acts_dia = defaultdict(list)
    for a in actividades: acts_dia[a["dia"]].append(a)
    n_y = pen_y = 0
    for acts_d in acts_dia.values():
        mapa = _box_por_bloque(asign, acts_d)
        bls  = sorted({t for (_, t) in mapa.keys()})
        if not bls: continue
        for prof in {a["profesional"] for a in acts_d}:
            w = W2 * (1 + Palt.get(prof, 0))
            for t in range(bls[0], bls[-1]):
                b1, b2 = mapa.get((prof, t)), mapa.get((prof, t + 1))
                if b1 is not None and b2 is not None and b1 != b2:
                    n_y += 1; pen_y += w
    return {"bloques_sin_box": b_sin, "pen_u": pen_u, "n_y": n_y, "pen_y": pen_y, "Z": -pen_u - pen_y}

# ── 8. Aporte marginal ─────────────────────────────────────────────────────────

def aporte_marginal(a, b, asign, acts_dia, Palt):
    aid, orig = a["id"], asign.get(a["id"])
    asign[aid] = None
    antes   = W1 * a["n_bloques"] + costo_estabilidad(asign, acts_dia, Palt)
    asign[aid] = b
    despues = costo_estabilidad(asign, acts_dia, Palt)
    asign[aid] = orig
    return antes - despues

# ── 9. Candidatos válidos ──────────────────────────────────────────────────────

def candidatos_validos(a, maestro, regla_map, ocupacion, cap_b):
    comba, b_medicos, prof_norm = maestro["comba"], maestro["b_medicos"], a["profesional_norm"]

    box_r, tipo_r = get_box_regla(prof_norm, a["tipo"], a["actividad"],
                                   a["dia"], a["bloques"], regla_map, comba)
    if box_r is not None:
        return ([box_r] if disponible(box_r, a, ocupacion, cap_b) else []), tipo_r

    if a["es_no_medico"]:
        pols_hab = maestro["hab_poli_nm"].get(a["nm_canon"], [])
        if pols_hab and a["policlinico"] not in pols_hab: return [], None
        return ([b for b in a["hab_boxes"]
                 if comba.get((b, a["tipo"]), 0) == 1 and disponible(b, a, ocupacion, cap_b)], None)

    poli = a["policlinico"]
    esp  = normalize(a["specialty"])
    if maestro["esp_poli"].get((esp, poli), 0) != 1: return [], None
    boxes_poli = maestro["b_poli"].get(poli, [])
    base_med   = boxes_poli if boxes_poli else b_medicos
    return ([b for b in base_med
              if b in b_medicos and comba.get((b, a["tipo"]), 0) == 1
              and disponible(b, a, ocupacion, cap_b)], None)

# ── 10. Fase 1 — Construcción greedy randomizada ──────────────────────────────

def _ordenar_por_profesional_cronologico(acts_dia):
    por_prof = defaultdict(list)
    for a in acts_dia: por_prof[a["profesional"]].append(a)
    for prof in por_prof: por_prof[prof].sort(key=lambda x: x["bloques"][0])
    profs = list(por_prof.keys()); random.shuffle(profs)
    resultado, indices = [], {p: 0 for p in profs}
    while True:
        avance = False
        for prof in profs:
            idx = indices[prof]
            if idx < len(por_prof[prof]):
                resultado.append(por_prof[prof][idx]); indices[prof] += 1; avance = True
        if not avance: break
    return resultado

def fase1(acts_dia, Palt, alpha, asign, ocupacion, maestro, regla_map, cap_b):
    for a in _ordenar_por_profesional_cronologico(acts_dia):
        aid = a["id"]
        cands, tipo_r = candidatos_validos(a, maestro, regla_map, ocupacion, cap_b)
        if not cands: asign[aid] = None; continue
        if tipo_r is not None:
            asign[aid] = cands[0]; ocupar(cands[0], a, ocupacion); continue
        aportes = [(b, aporte_marginal(a, b, asign, acts_dia, Palt)) for b in cands]
        g_max, g_min = max(s for _, s in aportes), min(s for _, s in aportes)
        rcl = ([b for b, _ in aportes] if g_max == g_min
               else [b for b, s in aportes if s >= g_max - alpha * (g_max - g_min)])
        box = random.choice(rcl)
        asign[aid] = box; ocupar(box, a, ocupacion)

# ── 11. Fase 2 — Búsqueda local ───────────────────────────────────────────────

def fase2(asign, acts_dia, Palt, ocupacion, maestro, regla_map, cap_b):
    comba = maestro["comba"]
    while True:
        costo_ini = costo_estabilidad(asign, acts_dia, Palt)
        mejor_gan, mejor_mov = 0.0, None
        sin_box = [a for a in acts_dia if asign.get(a["id"]) is None]
        con_box = [a for a in acts_dia if asign.get(a["id"]) is not None]

        # Relocate
        for a in sin_box + con_box:
            aid, b_act = a["id"], asign.get(a["id"])
            _, tipo_r = get_box_regla(a["profesional_norm"], a["tipo"], a["actividad"],
                                       a["dia"], a["bloques"], regla_map, comba)
            if tipo_r is not None: continue
            if b_act is not None: liberar(b_act, a, ocupacion)
            vecinos, _ = candidatos_validos(a, maestro, regla_map, ocupacion, cap_b)
            vecinos = [b for b in vecinos if b != b_act]
            if not vecinos:
                asign[aid] = b_act
                if b_act is not None: ocupar(b_act, a, ocupacion)
                continue
            gan_u = W1 * a["n_bloques"] if b_act is None else 0
            for b_new in vecinos:
                asign[aid] = b_new; ocupar(b_new, a, ocupacion)
                gan = (costo_ini - costo_estabilidad(asign, acts_dia, Palt)) + gan_u
                if gan > mejor_gan:
                    mejor_gan = gan; mejor_mov = ("relocate", aid, b_new, b_act)
                liberar(b_new, a, ocupacion); asign[aid] = b_act
            if b_act is not None: ocupar(b_act, a, ocupacion)
            else: asign[aid] = None

        # Swap
        medicos = [a for a in acts_dia if not a["es_no_medico"] and asign.get(a["id"]) is not None]
        for i in range(len(medicos)):
            for j in range(i + 1, len(medicos)):
                ai, aj = medicos[i], medicos[j]
                _, ri = get_box_regla(ai["profesional_norm"], ai["tipo"], ai["actividad"],
                                       ai["dia"], ai["bloques"], regla_map, comba)
                _, rj = get_box_regla(aj["profesional_norm"], aj["tipo"], aj["actividad"],
                                       aj["dia"], aj["bloques"], regla_map, comba)
                if ri is not None or rj is not None: continue
                bi, bj = asign[ai["id"]], asign[aj["id"]]
                if bi == bj: continue
                if not comba.get((bj, ai["tipo"]), 0): continue
                if not comba.get((bi, aj["tipo"]), 0): continue
                liberar(bi, ai, ocupacion); liberar(bj, aj, ocupacion)
                if disponible(bj, ai, ocupacion, cap_b) and disponible(bi, aj, ocupacion, cap_b):
                    asign[ai["id"]] = bj; asign[aj["id"]] = bi
                    ocupar(bj, ai, ocupacion); ocupar(bi, aj, ocupacion)
                    gan = costo_ini - costo_estabilidad(asign, acts_dia, Palt)
                    if gan > mejor_gan:
                        mejor_gan = gan; mejor_mov = ("swap", ai, aj, bi, bj)
                    liberar(bj, ai, ocupacion); liberar(bi, aj, ocupacion)
                asign[ai["id"]] = bi; asign[aj["id"]] = bj
                ocupar(bi, ai, ocupacion); ocupar(bj, aj, ocupacion)

        if mejor_mov is None or mejor_gan <= 0: break
        if mejor_mov[0] == "relocate":
            _, aid, b_new, b_old = mejor_mov
            act_obj = next(a for a in acts_dia if a["id"] == aid)
            if b_old is not None: liberar(b_old, act_obj, ocupacion)
            asign[aid] = b_new; ocupar(b_new, act_obj, ocupacion)
        else:
            _, ai, aj, bi, bj = mejor_mov
            liberar(bi, ai, ocupacion); liberar(bj, aj, ocupacion)
            asign[ai["id"]] = bj; asign[aj["id"]] = bi
            ocupar(bj, ai, ocupacion); ocupar(bi, aj, ocupacion)

# ── 12. Verificación y reparación ─────────────────────────────────────────────

def detectar_conflictos(asign, actividades, cap_b):
    box_check, prof_check = defaultdict(list), defaultdict(list)
    for a in actividades:
        box = asign.get(a["id"])
        if box is None: continue
        for t in a["bloques"]:
            box_check[ (a["dia"], t, box)].append(a["id"])
            prof_check[(a["dia"], t, a["profesional"])].append(a["id"])
    return ({k: ids for k, ids in box_check.items()  if len(ids) > cap_b.get(k[2], 1)},
            {k: ids for k, ids in prof_check.items() if len(ids) > 1})

def reparar(asign, actividades, Palt, cap_b):
    n_rep = 0
    por_id  = {a["id"]: a for a in actividades}
    por_dia = defaultdict(list)
    for a in actividades: por_dia[a["dia"]].append(a)
    for _ in range(10):
        c_box, c_prof = detectar_conflictos(asign, actividades, cap_b)
        if not c_box and not c_prof: break
        for (dia, t, box), ids in list(c_box.items()):
            act = [aid for aid in ids if asign.get(aid) == box]
            cap = cap_b.get(box, 1)
            if len(act) <= cap: continue
            srt = sorted([(aporte_marginal(por_id[aid], box, asign, por_dia[dia], Palt), aid)
                           for aid in act], reverse=True)
            for _, aid_d in srt[cap:]:
                if asign.get(aid_d) is not None: asign[aid_d] = None; n_rep += 1
        c_box, c_prof = detectar_conflictos(asign, actividades, cap_b)
        for (dia, t, prof), ids in list(c_prof.items()):
            act = [aid for aid in ids if asign.get(aid) is not None]
            if len(act) <= 1: continue
            srt = sorted([(aporte_marginal(por_id[aid], asign[aid], asign, por_dia[dia], Palt), aid)
                           for aid in act if asign.get(aid) is not None], reverse=True)
            for _, aid_d in srt[1:]:
                if asign.get(aid_d) is not None: asign[aid_d] = None; n_rep += 1
    return n_rep

# ── 13. Bucle GRASP ────────────────────────────────────────────────────────────

def barrido_alpha(actividades, Palt, maestro, regla_map, cap_b, n_ciclos=N, alphas_lista=None):
    if alphas_lista is None:
        alphas_lista = ALPHAS          # comportamiento por defecto inalterado
    por_dia = defaultdict(list)
    for a in actividades: por_dia[a["dia"]].append(a)
    mejor_asign, mejor_Z, mejor_alpha = None, float("-inf"), None
    registro = {a: [] for a in alphas_lista}
    print(f"\n  GRASP | {len(alphas_lista)} α × {n_ciclos} iter. = {len(alphas_lista)*n_ciclos} total")
    print(f"\n  {'α':>4}  {'ITER':>5}  {'Z iter':>10}  {'Mejor Z':>10}")
    print(f"  {'─'*4}  {'─'*5}  {'─'*10}  {'─'*10}")

    for alpha in alphas_lista:
        for ciclo in range(1, n_ciclos + 1):
            random.seed()          # entropía del SO → no determinístico
            asign, ocu = {a["id"]: None for a in actividades}, {}
            for dia in DIAS_VALIDOS:
                acts_d = por_dia.get(dia, [])
                if not acts_d: continue
                fase1(acts_d, Palt, alpha, asign, ocu, maestro, regla_map, cap_b)
                fase2(asign, acts_d, Palt, ocu, maestro, regla_map, cap_b)
            cb, cp = detectar_conflictos(asign, actividades, cap_b)
            if cb or cp: reparar(asign, actividades, Palt, cap_b)
            Z_i = calcular_Z(asign, actividades, Palt)
            registro[alpha].append(Z_i)
            if Z_i > mejor_Z:
                mejor_Z = Z_i; mejor_asign = dict(asign); mejor_alpha = alpha
            print(f"  {alpha:>4.1f}  {ciclo:>5}  {Z_i:>10.1f}  {mejor_Z:>10.1f}"
                  + (" ◄" if Z_i == mejor_Z else ""))
        print(f"  {'─'*4}  {'─'*5}  {'─'*10}  {'─'*10}")
    return mejor_asign, mejor_Z, mejor_alpha, registro

# ── 14. Gráfico ────────────────────────────────────────────────────────────────

def graficar(registro_alpha, mejor_Z, alpha_opt):
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    alphas = sorted(registro_alpha.keys())
    medias = [statistics.mean(registro_alpha[a]) for a in alphas]
    AZUL, NARAN = "#1F3864", "#C55A11"
    plt.rcParams.update({"font.family": "serif", "axes.spines.top": False,
                          "axes.spines.right": False, "axes.grid": True,
                          "grid.alpha": 0.25, "grid.linestyle": "--"})
    fig, ax = plt.subplots(figsize=(11, 5))
    bp = ax.boxplot([registro_alpha[a] for a in alphas], positions=range(len(alphas)),
                    patch_artist=True, widths=0.55,
                    medianprops=dict(color="white", lw=2),
                    whiskerprops=dict(color=AZUL, lw=1.2), capprops=dict(color=AZUL, lw=1.2),
                    flierprops=dict(marker="o", markerfacecolor=NARAN, markersize=4, linestyle="none"))
    for patch in bp["boxes"]: patch.set_facecolor(NARAN); patch.set_alpha(0.75)
    for i, m in enumerate(medias):
        ax.plot(i, m, "D", color="white", markeredgecolor=AZUL, ms=7, zorder=5,
                label="Media" if i == 0 else "")
    idx_opt = alphas.index(alpha_opt)
    ax.axvline(idx_opt, color=AZUL, lw=2, ls=":", label=f"α* = {alpha_opt}")
    ax.annotate(f"α* = {alpha_opt}\nMedia = {medias[idx_opt]:.0f}",
                xy=(idx_opt, medias[idx_opt]), xytext=(idx_opt + 0.6, medias[idx_opt] - 25),
                fontsize=9, color=AZUL, arrowprops=dict(arrowstyle="->", color=AZUL))
    ax.axhline(mejor_Z, color="red", lw=1.2, ls="--", alpha=0.6,
               label=f"Mejor Z global = {mejor_Z:.0f}")
    ax.set_xticks(range(len(alphas))); ax.set_xticklabels([str(a) for a in alphas])
    ax.set_xlabel("Valor de α", fontsize=12)
    ax.set_ylabel("Distribución de Z  (◆ = media)", fontsize=12)
    ax.set_title("GRASP — Distribución de Z por α\nHospital Guillermo Grant Benavente",
                 fontsize=13, fontweight="bold", color=AZUL)
    ax.legend(fontsize=9, framealpha=0.9); plt.tight_layout()
    ruta = "grasp_B_v4_corregido_boxplot.png"
    plt.savefig(ruta, dpi=180, bbox_inches="tight"); plt.close()
    print(f"  ✓ {ruta}"); return ruta

# ── 15. Exportar Excel ─────────────────────────────────────────────────────────

def _cell(ws, row, col, val=None, *, bold=False, fc="000000", sz=10,
          bg=None, ha="center", wrap=True):
    """Escribe celda con todos sus estilos en una sola llamada."""
    c = ws.cell(row=row, column=col, value=val)
    c.font = Font(bold=bold, color=fc, size=sz)
    if bg: c.fill = PatternFill("solid", fgColor=bg)
    c.alignment = Alignment(horizontal=ha, vertical="center", wrap_text=wrap)
    s = Side(style="thin", color="BFBFBF")
    c.border = Border(left=s, right=s, top=s, bottom=s)
    return c

def _profs_slot(lista, ini, fin):
    seen, found = set(), []
    for a in lista:
        if a["desde_min"] < fin and a["hasta_min"] > ini:
            ap = " ".join(a["profesional"].strip().split()[:2])
            if ap not in seen: seen.add(ap); found.append(ap)
    return " / ".join(found) if found else None

def exportar(actividades, asign, Palt, maestro, regla_map, ruta):
    labels           = maestro["labels"]
    box_ids          = maestro["box_ids"]
    boxes_excl_nm    = maestro["boxes_excl_nm"]
    boxes_excl_regla = maestro["boxes_excl_regla"]
    boxes_cons = [b for b in box_ids if b not in boxes_excl_nm and b not in boxes_excl_regla]
    boxes_nm   = [b for b in box_ids if b in boxes_excl_nm]
    boxes_esp  = [b for b in box_ids if b in boxes_excl_regla]

    wb = openpyxl.Workbook()
    ws1 = wb.active; ws1.title = "Distribución de Boxes"

    col_h = 1; col_c_ini = 2; col_c_fin = col_c_ini + len(boxes_cons) - 1
    col_sep1 = col_c_fin + 1; col_n_ini = col_sep1 + 1
    col_n_fin = col_n_ini + len(boxes_nm)  - 1 if boxes_nm  else col_sep1
    col_sep2  = col_n_fin + 1; col_e_ini = col_sep2 + 1
    col_e_fin = col_e_ini + len(boxes_esp) - 1 if boxes_esp else col_sep2
    n_cols = max(col_c_fin, col_n_fin if boxes_nm else col_c_fin, col_e_fin if boxes_esp else col_c_fin)

    ws1.column_dimensions["A"].width = 17
    for c in range(2, n_cols + 1): ws1.column_dimensions[get_column_letter(c)].width = 15

    ws1.merge_cells(start_row=1, start_column=1, end_row=1, end_column=n_cols)
    _cell(ws1, 1, 1, "DISTRIBUCIÓN DE POLICLÍNICOS — GRASP-B v4 CORREGIDO",
          bold=True, fc="FFFFFF", sz=13, bg="1F3864")
    ws1.row_dimensions[1].height = 26

    para_box = defaultdict(list)
    for a in actividades:
        box = asign.get(a["id"])
        if box: para_box[(a["dia"], box)].append(a)

    fila = 2; SLOT = 60
    for dia in DIAS_VALIDOS:
        ws1.merge_cells(start_row=fila, start_column=1, end_row=fila, end_column=n_cols)
        _cell(ws1, fila, 1, dia, bold=True, fc="FFFFFF", sz=12, bg="2E75B6")
        ws1.row_dimensions[fila].height = 22; fila += 1

        _cell(ws1, fila, col_h, "HORARIO", bold=True, fc="FFFFFF", sz=10, bg="1F3864")
        for i, box in enumerate(boxes_cons):
            _cell(ws1, fila, col_c_ini+i, labels.get(box, box), bold=True, fc="FFFFFF", sz=10, bg="2E75B6")
        if boxes_nm:
            for i, box in enumerate(boxes_nm):
                _cell(ws1, fila, col_n_ini+i, labels.get(box, box), bold=True, fc="FFFFFF", sz=10, bg="375623")
        if boxes_esp:
            for i, box in enumerate(boxes_esp):
                _cell(ws1, fila, col_e_ini+i, labels.get(box, box), bold=True, fc="7F6000", sz=10, bg="FFF2CC")
        ws1.row_dimensions[fila].height = 30; fila += 1

        slot = DIA_INICIO_MIN
        while slot < DIA_FIN_MIN:
            fin   = slot + SLOT
            fondo = "F2F2F2" if (slot // 60) % 2 == 0 else "FFFFFF"
            _cell(ws1, fila, col_h, f"{_hhmm(slot)}-{_hhmm(fin)}", bold=True, sz=9, bg="BDD7EE")
            for i, box in enumerate(boxes_cons):
                txt = _profs_slot(para_box.get((dia, box), []), slot, fin)
                _cell(ws1, fila, col_c_ini+i, txt, sz=9, bg="E2EFDA" if txt else fondo)
            if boxes_nm:
                ws1.cell(row=fila, column=col_sep1).fill = PatternFill("solid", fgColor="FFFFFF")
                for i, box in enumerate(boxes_nm):
                    txt = _profs_slot(para_box.get((dia, box), []), slot, fin)
                    _cell(ws1, fila, col_n_ini+i, txt, sz=9, bg="EBF3FB" if txt else fondo)
            if boxes_esp:
                ws1.cell(row=fila, column=col_sep2).fill = PatternFill("solid", fgColor="FFFFFF")
                for i, box in enumerate(boxes_esp):
                    txt = _profs_slot(para_box.get((dia, box), []), slot, fin)
                    _cell(ws1, fila, col_e_ini+i, txt, sz=9, bg="FFF2CC" if txt else fondo)
            ws1.row_dimensions[fila].height = 18
            slot += SLOT; fila += 1
        fila += 1

    # Hoja Sin Box
    ws2 = wb.create_sheet("Sin Box Asignado")
    COLS2 = [("PROFESIONAL",32), ("PROFESIÓN",18), ("TIPO PROF.",11), ("POLICLÍNICO",20),
             ("DÍA",12), ("ACTIVIDAD",32), ("TIPO",8), ("DESDE",8), ("HASTA",8), ("BLOQUES SIN CUBRIR",22)]
    for cidx, (txt, w) in enumerate(COLS2, 1):
        _cell(ws2, 1, cidx, txt, bold=True, fc="FFFFFF", sz=11, bg="1F3864")
        ws2.column_dimensions[get_column_letter(cidx)].width = w
    ws2.row_dimensions[1].height = 22

    sin_box = sorted([a for a in actividades if asign.get(a["id"]) is None],
                     key=lambda a: (DIAS_VALIDOS.index(a["dia"]) if a["dia"] in DIAS_VALIDOS else 9,
                                    a["profesional"], a["desde_min"]))
    if not sin_box:
        ws2.cell(row=2, column=1, value="✅ Todos los profesionales tienen box asignado")
    else:
        for f2, a in enumerate(sin_box, 2):
            vals = [a["profesional"], a["profesion"], "No médico" if a["es_no_medico"] else "Médico",
                    a["policlinico"], a["dia"], a["actividad"], a["tipo"],
                    _hhmm(a["desde_min"]), _hhmm(a["hasta_min"]),
                    f"{_hhmm(a['desde_min'])} → {_hhmm(a['hasta_min'])} ({a['n_bloques']} bloques)"]
            color = "EBF3FB" if a["es_no_medico"] else "FFCCCC"
            for cidx, val in enumerate(vals, 1):
                _cell(ws2, f2, cidx, val, sz=10, bg=color,
                      ha="left" if cidx in (1, 2, 4, 6) else "center")
            ws2.row_dimensions[f2].height = 18
    ws2.freeze_panes = "A2"

    # Hoja Resumen
    ws3 = wb.create_sheet("Resumen")
    res = desglose_Z(asign, actividades, Palt)
    FILAS = [("Indicador","Valor"), ("Actividades totales",len(actividades)),
             ("Actividades sin box",len(sin_box)), ("Bloques sin box",res["bloques_sin_box"]),
             ("Penalización cobertura",res["pen_u"]), ("Cambios de box",res["n_y"]),
             ("Penalización cambio box",res["pen_y"]), ("Z",res["Z"])]
    for r, (k, v) in enumerate(FILAS, 1):
        hdr = r == 1
        _cell(ws3, r, 1, k, ha="left",   bold=hdr, fc="FFFFFF" if hdr else "000000", bg="1F3864" if hdr else None)
        _cell(ws3, r, 2, v, ha="center", bold=hdr, fc="FFFFFF" if hdr else "000000", bg="1F3864" if hdr else None)
    ws3.column_dimensions["A"].width = 30; ws3.column_dimensions["B"].width = 18

    wb.save(ruta); print(f"  ✓ {ruta}")

# ── 16. Simulación de ausencia ─────────────────────────────────────────────────

def resolver_profesional(nombre_buscado, actividades):
    objetivo = normalize(nombre_buscado)
    nombres  = {a["profesional_norm"]: a["profesional"] for a in actividades}
    if objetivo in nombres: return nombres[objetivo]
    matches = sorted({real for norm, real in nombres.items() if objetivo in norm})
    if len(matches) == 1: return matches[0]
    if len(matches) > 1:
        raise ValueError(f"'{nombre_buscado}' coincide con varios: {matches}")
    raise ValueError(f"No se encontró '{nombre_buscado}' en la agenda.")

def simular_ausencia_profesional(
    profesional_ausente, actividades, asign_base, maestro, regla_map,
    cap_b, Palt, dia_ausencia=None,
):
    profesional_ausente = resolver_profesional(profesional_ausente, actividades)
    prof_norm = normalize(profesional_ausente)
    dia_norm  = normalize(dia_ausencia) if dia_ausencia else None

    asign_sim = dict(asign_base)
    ocu_sim   = reconstruir_ocupacion(asign_sim, actividades)

    acts_ausente = [a for a in actividades
                    if a["profesional_norm"] == prof_norm
                    and (dia_norm is None or a["dia"] == dia_norm)]
    if not acts_ausente:
        raise ValueError(f"'{profesional_ausente}' sin actividades"
                         + (f" el {dia_ausencia}." if dia_ausencia else "."))

    ids_ausente = {a["id"] for a in acts_ausente}
    espacios_liberados = []
    for a in acts_ausente:
        box_a = asign_sim.get(a["id"])
        if box_a is not None:
            liberar(box_a, a, ocu_sim)
            espacios_liberados.append({"actividad": a, "box": box_a})
        asign_sim[a["id"]] = None

    dias_afect = {e["actividad"]["dia"] for e in espacios_liberados}
    candidatos = sorted(
        [a for a in actividades if a["id"] not in ids_ausente
         and asign_base.get(a["id"]) is None and a["dia"] in dias_afect],
        key=lambda a: (DIAS_VALIDOS.index(a["dia"]), a["desde_min"])
    )
    por_dia = defaultdict(list)
    for a in actividades: por_dia[a["dia"]].append(a)

    reasignadas = []
    for a in candidatos:
        cands, tipo_r = candidatos_validos(a, maestro, regla_map, ocu_sim, cap_b)
        if not cands: continue
        if tipo_r is not None:
            box_e = cands[0]
        else:
            aportes = [(b, aporte_marginal(a, b, asign_sim, por_dia[a["dia"]], Palt)) for b in cands]
            box_e   = max(aportes, key=lambda x: x[1])[0]
        asign_sim[a["id"]] = box_e; ocupar(box_e, a, ocu_sim)
        reasignadas.append({"actividad": a, "box": box_e})

    ids_reas   = {r["actividad"]["id"] for r in reasignadas}
    aun_sin    = [a for a in candidatos if a["id"] not in ids_reas]
    acts_ef    = [a for a in actividades if a["id"] not in ids_ausente]
    asign_b_ef = {k: v for k, v in asign_base.items() if k not in ids_ausente}
    Z_antes    = calcular_Z(asign_b_ef, acts_ef, Palt)
    Z_despues  = calcular_Z(asign_sim,  acts_ef, Palt)
    no_altero  = all(asign_sim[a["id"]] == asign_base[a["id"]]
                     for a in actividades
                     if a["id"] not in ids_ausente and a["id"] not in ids_reas)
    return {
        "profesional": profesional_ausente, "dia": dia_ausencia,
        "asign_simulada": asign_sim, "espacios_liberados": espacios_liberados,
        "reasignadas": reasignadas, "aun_sin_box": aun_sin,
        "Z_antes": Z_antes, "Z_despues": Z_despues,
        "no_altero_otras_asignaciones": no_altero,
    }

def reportar_simulacion_ausencia(resultado):
    prof = resultado["profesional"]
    dia  = resultado["dia"] or "TODA LA SEMANA"
    sep  = "=" * 60
    print(f"{sep}\nSIMULACIÓN DE AUSENCIA — {prof} ({dia})\n{sep}")

    print(f"\nEspacios liberados: {len(resultado['espacios_liberados'])}")
    for e in resultado["espacios_liberados"]:
        a = e["actividad"]
        print(f"  {a['dia']:<10} {e['box']:<8} {_hhmm(a['desde_min'])}-{_hhmm(a['hasta_min'])}  {a['actividad']}")

    print(f"\nActividades sin box → ahora cubiertas: {len(resultado['reasignadas'])}")
    if not resultado["reasignadas"]:
        print("  (ninguna: no calzaron horarios o compatibilidad de box)")
    for r in resultado["reasignadas"]:
        a = r["actividad"]
        print(f"  {a['profesional']:<30} {a['dia']:<10} "
              f"{_hhmm(a['desde_min'])}-{_hhmm(a['hasta_min'])}  →  {r['box']:<8} ({a['actividad']})")

    print(f"\nActividades que siguen SIN box: {len(resultado['aun_sin_box'])}")
    for a in resultado["aun_sin_box"]:
        print(f"  {a['profesional']:<30} {a['dia']:<10} "
              f"{_hhmm(a['desde_min'])}-{_hhmm(a['hasta_min'])}  ({a['actividad']})")

    print(f"\n  Z (sin ausente) ANTES  : {resultado['Z_antes']:.1f}")
    print(f"  Z (sin ausente) DESPUÉS: {resultado['Z_despues']:.1f}")
    ok = "✅ Sí" if resultado["no_altero_otras_asignaciones"] else "❌ NO — revisar"
    print(f"  ¿Demás asignaciones intactas? {ok}")

# ── 17. MAIN ───────────────────────────────────────────────────────────────────

if __name__ == "__main__":
    print("GRASP — Hospital Guillermo Grant Benavente\n")

    maestro = load_maestro(RUTA_MAESTRO)
    cap_b   = maestro["cap_b"]

    actividades, Palt, delta, P_M, P_NM, name_norm_to_prof = leer_agenda(RUTA_EXCEL, maestro)
    n_med = sum(1 for a in actividades if not a["es_no_medico"])
    n_nm  = sum(1 for a in actividades if     a["es_no_medico"])
    print(f"  {len(actividades)} actividades ({n_med} médicas | {n_nm} NM) | "
          f"delta={delta} min | {len(P_M)} médicos | {len(P_NM)} NM")

    # Validación especialidades (8.9)
    esp_usadas = {normalize(a["specialty"]) for a in actividades if not a["es_no_medico"]}
    sin_reg  = [e for e in esp_usadas if e not in maestro["especialidades_canon"]]
    sin_poli = [e for e in esp_usadas if e in maestro["especialidades_canon"]
                and not any(maestro["esp_poli"].get((e, rho)) == 1 for rho in maestro["poli_ids"])]
    if sin_reg:  print(f"  [AVISO] Especialidades sin registrar en Especialidades_Poli: {sorted(sin_reg)}")
    if sin_poli: print(f"  [AVISO] Especialidades sin policlínico habilitado: {sorted(sin_poli)}")

    regla_map = build_regla_map(maestro["reglas"], name_norm_to_prof, delta)
    if not regla_map:
        print("  Sin reglas especiales.")
    else:
        for r in regla_map:
            print(f"  Regla {r['id']} ({r['tipo']}): {r['profesional']} | "
                  f"{r['dia']} | act='{r['actividad_norm']}' | bloques={sorted(r['blocks'])} → {r['box']}")

    # ── Selección de modo de ejecución ────────────────────────────────────────
    print("\n  ┌─────────────────────────────────────────┐")
    print("  │         MODO DE EJECUCIÓN GRASP         │")
    print("  ├─────────────────────────────────────────┤")
    print("  │  [1]  Alpha fijo  (un único valor de α) │")
    print("  │  [2]  Barrido completo  (todos los α)   │")
    print("  └─────────────────────────────────────────┘")

    while True:
        opcion = input("  Seleccione opción [1/2]: ").strip()
        if opcion in {"1", "2"}:
            break
        print("  ⚠  Ingrese 1 ó 2.")

    if opcion == "1":
        print(f"  Valores disponibles: {ALPHAS}")
        while True:
            try:
                alpha_ingresado = round(float(input("  Ingrese el valor de α: ").strip()), 1)
                if alpha_ingresado in ALPHAS:
                    alphas_usar = [alpha_ingresado]
                    print(f"  ✓ Ejecutando con α fijo = {alpha_ingresado}")
                    break
                print(f"  ⚠  α debe ser uno de {ALPHAS}")
            except ValueError:
                print("  ⚠  Valor inválido, ingrese un número decimal.")
    else:
        alphas_usar = ALPHAS
        print(f"  ✓ Ejecutando barrido completo: {ALPHAS}")

    mejor_asign, mejor_Z, mejor_alpha, registro_alpha = barrido_alpha(
        actividades, Palt, maestro, regla_map, cap_b, n_ciclos=N, alphas_lista=alphas_usar)

    # Verificación reglas
    if regla_map:
        print("\nVerificación de reglas:")
        for r in regla_map:
            actos = [a for a in actividades
                     if a["profesional_norm"] == r["prof_norm"]
                     and (r["dia"] == "*" or a["dia"] == r["dia"])
                     and (set(a["bloques"]) & r["blocks"])
                     and (r["actividad_norm"] == "*" or r["actividad_norm"] in a["actividad_norm"])]
            ok = sum(1 for a in actos if mejor_asign.get(a["id"]) == r["box"])
            print(f"  Regla {r['id']}: {'✅' if ok == len(actos) else f'⚠️ {ok}/{len(actos)}'}")

    # Factibilidad
    c_box, c_prof = detectar_conflictos(mejor_asign, actividades, cap_b)
    print(f"\n  R8.3: {'✅' if not c_box  else f'❌ {len(c_box)}  conflictos'}"
          f"  |  R8.4: {'✅' if not c_prof else f'❌ {len(c_prof)} conflictos'}")
    if c_box or c_prof:
        n_rep = reparar(mejor_asign, actividades, Palt, cap_b)
        mejor_Z = calcular_Z(mejor_asign, actividades, Palt)
        c_box, c_prof = detectar_conflictos(mejor_asign, actividades, cap_b)
        print(f"  Reparación: {n_rep} desasignadas | Z={mejor_Z:.1f} | "
              f"R8.3: {'✅' if not c_box else '❌'} | R8.4: {'✅' if not c_prof else '❌'}")

    # Desglose Z
    res     = desglose_Z(mejor_asign, actividades, Palt)
    sin_box = [a for a in actividades if mejor_asign.get(a["id"]) is None]
    print(f"\n  Z={res['Z']:.1f} | bloques sin box={res['bloques_sin_box']} (pen={res['pen_u']}) | "
          f"cambios box={res['n_y']} (pen={res['pen_y']:.0f}) | "
          f"act. sin box={len(sin_box)} | α*={mejor_alpha}")

    exportar(actividades, mejor_asign, Palt, maestro, regla_map, "Resultado_GRASP.xlsx")
    alphas    = sorted(registro_alpha.keys())
    medias    = [statistics.mean(registro_alpha[a]) for a in alphas]
    alpha_opt = alphas[medias.index(max(medias))]
    graficar(registro_alpha, mejor_Z, alpha_opt)
    print("\n✅ Terminado.")

In [ ]:
# Requiere haber ejecutado antes la celda principal.

print("¿Qué situación quieres trabajar?")
print("  1) Situación normal (la solución ya calculada por el GRASP)")
print("  2) Simular la ausencia de un profesional puntual")
opcion = input("Ingresa 1 o 2: ").strip()

if opcion == "2":
    nombre  = input("Nombre del profesional ausente (como aparece en la agenda): ").strip()
    dia_txt = input(
        "Día de la ausencia (LUNES/MARTES/MIERCOLES/JUEVES/VIERNES) "
        "o ENTER para toda la semana: "
    ).strip()
    dia_ausencia = dia_txt if dia_txt else None

    try:
        resultado_ausencia = simular_ausencia_profesional(
            profesional_ausente=nombre,
            actividades=actividades,
            asign_base=mejor_asign,
            maestro=maestro,
            regla_map=regla_map,
            cap_b=cap_b,
            Palt=Palt,
            dia_ausencia=dia_ausencia,
        )
        reportar_simulacion_ausencia(resultado_ausencia)
  
    except ValueError as e:
        print(f"\n  {e}")

else:
    print("\n" + "=" * 60)
    print("SITUACIÓN NORMAL — resultados del GRASP ya ejecutado")
    print("=" * 60)
    res_normal = desglose_Z(mejor_asign, actividades, Palt)
    sin_box    = [a for a in actividades if mejor_asign.get(a["id"]) is None]
    print(f"  α (mejor Z global)   : {mejor_alpha}")
    print(f"  Mejor Z              : {mejor_Z:.1f}")
    print(f"  Bloques sin box (u)  : {res_normal['bloques_sin_box']}  → -{res_normal['pen_u']}")
    print(f"  Cambios de box (y)   : {res_normal['n_y']}  → -{res_normal['pen_y']}")
    print(f"  Actividades sin box  : {len(sin_box)}")